# Ensemble Learning (Bagging & Boosting) — GenAI-Relevant Notes

**Why this matters for GenAI engineering:**
- You won't be building Random Forests inside an LLM pipeline, but ensemble concepts (bias/variance, sequential vs parallel learning, regularization) show up constantly in ML reasoning, evaluation design, and interview conversations.
- XGBoost/LightGBM remain the go-to for **tabular baselines** — useful if a GenAI project has a structured-data component (e.g. ranking retrieved docs, scoring/classification heads, feature-based rerankers).
- Goal here: **conceptual fluency + a working code template you can reuse**, not deep hyperparameter mastery.

**Structure of this notebook:**
1. Ensemble Learning — the big picture
2. Bagging vs Boosting (the one distinction to never forget)
3. Random Forest — quick reference + code
4. Gradient Boosting → XGBoost — the important one
5. LightGBM & CatBoost — when to reach for them
6. Evaluation templates (classification + regression)
7. One-page cheat sheet

## 1. Ensemble Learning — The Big Picture

Instead of one model making a prediction, combine multiple models:

```text
Dataset → [Model 1, Model 2, Model 3] → Combine → Final Prediction
```

Two families:

| | Bagging | Boosting |
|---|---|---|
| Training | Parallel / independent | Sequential |
| Fixes | Variance (instability) | Bias (underfitting/loss) |
| Example | Random Forest | XGBoost, LightGBM, CatBoost |

## 2. Bagging vs Boosting — The Core Mental Model

**Bagging (Bootstrap Aggregating)**
```text
Dataset → Bootstrap samples (random, with replacement) → Independent models → Average/Vote → Reduce VARIANCE
```

**Boosting**
```text
Dataset → Model 1 → learn from errors → Model 2 → learn from remaining errors → ... → Reduce BIAS/LOSS
```

**One-line versions (interview-ready):**
- *Bagging*: trains models independently on bootstrap samples and averages them to reduce variance.
- *Boosting*: trains models sequentially, each correcting the previous ensemble's errors, to reduce bias.

**Practical takeaway:** if a single model overfits wildly → try bagging (Random Forest). If a single model underfits / plateaus → try boosting (XGBoost).

## 3. Random Forest — Quick Reference

Bootstrap sampling + random feature selection + many decision trees + aggregation.

Good default baseline for tabular data — barely needs tuning, no feature scaling required.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Classification
clf = RandomForestClassifier(
    n_estimators=200,      # number of trees
    max_depth=None,        # let trees grow fully (control via min_samples_leaf if needed)
    max_features="sqrt",   # features considered per split
    n_jobs=-1,              # use all CPU cores
    random_state=42
)

# Regression (same idea)
reg = RandomForestRegressor(
    n_estimators=200,
    max_features=1.0,
    n_jobs=-1,
    random_state=42
)

# clf.fit(X_train, y_train)
# y_pred = clf.predict(X_test)

## 4. Gradient Boosting → XGBoost ⭐

**Gradient Boosting core idea:** each new tree is trained to move predictions in the direction that reduces the loss function (for regression w/ squared error, this is roughly "predict the residual").

**XGBoost = Extreme Gradient Boosting** — optimized + regularized gradient boosting. This is the one worth actually knowing well if you ever touch tabular ML.

In [ ]:
# !pip install xgboost --break-system-packages
from xgboost import XGBClassifier, XGBRegressor

# The parameters actually worth understanding (master these, skip the rest):
clf = XGBClassifier(
    n_estimators=100,       # how many trees
    learning_rate=0.1,      # how much each tree contributes (lower = need more trees)
    max_depth=3,            # how complex each tree is
    subsample=0.8,          # fraction of rows used per tree
    colsample_bytree=0.8,   # fraction of features used per tree
    random_state=42
)

reg = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# clf.fit(X_train, y_train)
# y_pred = clf.predict(X_test)

**Key relationship to remember:**
```text
Lower learning_rate  +  More n_estimators  =  Slower but more controlled learning
```

**Overfitting controls (in order of what to try first):** lower `max_depth` → lower `learning_rate` (+ more `n_estimators`) → `subsample`/`colsample_bytree` < 1.0 → regularization (`reg_alpha`, `reg_lambda`).

## 5. LightGBM & CatBoost — When to Reach for Them

You don't need to master these — just know **when** they're the right tool:

| Situation | Use |
|---|---|
| Large dataset, training speed matters | **LightGBM** (histogram-based, leaf-wise growth) |
| Lots of categorical features, want to avoid manual encoding | **CatBoost** (`cat_features=[...]`) |
| Standard tabular baseline, well-supported, tunable | **XGBoost** |

In [ ]:
# LightGBM — same parameter logic as XGBoost
# from lightgbm import LGBMClassifier
# model = LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=31, random_state=42)

# CatBoost — pass categorical columns directly, no one-hot encoding needed
# from catboost import CatBoostClassifier
# model = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, verbose=0, random_seed=42)
# model.fit(X_train, y_train, cat_features=["company", "fuel_type"])

## 6. Evaluation Templates

In [ ]:
# Classification evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_classifier(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

In [ ]:
# Regression evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_regressor(model, X_test, y_test):
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)
    print("MAE  :", mae)
    print("MSE  :", mse)
    print("RMSE :", rmse)
    print("R2   :", r2)

## 7. One-Page Cheat Sheet

```text
ENSEMBLE LEARNING
│
├── BAGGING (parallel, reduces VARIANCE)
│   └── Random Forest ⭐ — bootstrap + random features + many trees
│
└── BOOSTING (sequential, reduces BIAS/LOSS)
    ├── AdaBoost — reweight difficult samples
    ├── Gradient Boosting — reduce loss step by step
    ├── XGBoost ⭐⭐⭐ — optimized + regularized GB, THE one to know
    ├── LightGBM — fast, for large datasets
    └── CatBoost — best for categorical-heavy data
```

**5 things worth actually remembering for GenAI-adjacent work:**
1. Bagging = independent + averaging → lower variance. Boosting = sequential + error-correcting → lower bias.
2. Tree-based ensembles don't need feature scaling (unlike KNN/Logistic Regression/Neural Nets).
3. XGBoost's core knobs: `n_estimators`, `learning_rate`, `max_depth`, `subsample`, `colsample_bytree`.
4. `learning_rate` ↓ generally requires `n_estimators` ↑.
5. Always compare against a simple baseline (Logistic/Linear Regression) before trusting a boosted model's gains — complexity should earn its keep.